# TCC — Experimentos YOLO26 no Google Colab

Notebook dedicado aos seis experimentos: detecção e segmentação, cada um em Base, LoRA e Full fine-tune. Também inclui smoke tests com YOLO26n.

Antes de executar: selecione GPU no Colab e configure `GITHUB_TOKEN` e `ROBOFLOW_API_KEY` em Secrets.


In [ ]:
import os, sys, subprocess, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', p.name)
    print('VRAM:', round(p.total_memory / 1024**3, 2), 'GB')


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
drive.mount('/content/drive')
RUNTIME = Path('/content/drive/MyDrive/TCC_Code_Test_runtime')
for name in ['data', 'workspace', 'exports', 'weights']:
    (RUNTIME / name).mkdir(parents=True, exist_ok=True)
print('Persistência:', RUNTIME)


In [ ]:
# Clona/atualiza o TCC privado sem imprimir o token.
from pathlib import Path
REPO = Path('/content/TCC_Code_Test')
token = userdata.get('GITHUB_TOKEN')
netrc = Path.home() / '.netrc'
netrc.write_text(f'machine github.com\nlogin x-access-token\npassword {token}\n', encoding='utf-8')
netrc.chmod(0o600)
try:
    if REPO.exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/ferreis/TCC_Code_Test.git', str(REPO)], check=True)
finally:
    netrc.unlink(missing_ok=True)
os.chdir(REPO)
print('TCC commit:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


In [ ]:
%%capture
!pip install -r requirements-colab.txt


In [ ]:
# Liga dados/resultados ao Drive.
import shutil
repo = Path('/content/TCC_Code_Test')
for name in ['data', 'workspace', 'exports', 'weights']:
    target = RUNTIME / name
    link = repo / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if link.is_dir():
            shutil.rmtree(link)
        else:
            link.unlink()
    link.symlink_to(target, target_is_directory=True)
print('Drive conectado ao projeto.')


In [ ]:
# Roboflow e dataset de detecção.
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
os.environ['ROBOFLOW_WORKSPACE'] = 'rafael-fernando-dos-reis-mecabo'
os.environ['ROBOFLOW_PROJECT'] = 'detecao-de-cetaceos-afrzs-iqey9'
os.environ['ROBOFLOW_VERSION'] = '1'
images_dir = Path('data/roboflow/train/images')
if not (images_dir.exists() and any(images_dir.iterdir())):
    subprocess.run([sys.executable, 'scripts/download_roboflow.py', '--format', 'yolo26', '--output', 'data/roboflow', '--overwrite'], check=True)
else:
    print('Dataset já existe:', images_dir)


## Backend LoRA

A Ultralytics oficial fornece YOLO26, porém o experimento LoRA usa o fork Tencent/YOLO-Master. Para manter a comparação consistente, rode os seis experimentos nesta mesma sessão após instalar o fork.


In [ ]:
!python scripts/setup_yolo_master.py --dir /content/YOLO-Master
print('Se esta célula substituir um pacote ultralytics já importado, reinicie o runtime e reexecute as células de setup antes de treinar.')


In [ ]:
# Confere qual implementação será usada.
import ultralytics, importlib.util
print('Ultralytics/YOLO-Master:', ultralytics.__version__)
print('Caminho:', ultralytics.__file__)
print('LoRA disponível:', importlib.util.find_spec('ultralytics.utils.lora') is not None)


## 1. Listar e validar a matriz

A matriz principal usa YOLO26n para reduzir custo durante a primeira rodada. `Base` congela o backbone, `LoRA` treina adaptadores e `Full` libera todos os pesos.


In [ ]:
!python scripts/train_experiments.py --group all --list
!python scripts/train_experiments.py --group experiments --dry-run


## 2. Smoke test YOLO26n — detecção

Roda 3 épocas para confirmar CUDA, dataset, checkpoint, VRAM e logging antes da bateria longa.


In [ ]:
!python scripts/train_experiments.py --group smoke_tests --only detect_yolo26n_smoke


## 3. Três experimentos de detecção

Base → LoRA → Full fine-tune. Todos usam o mesmo split, seed, resolução e protocolo global.


In [ ]:
!python scripts/train_experiments.py --only detect_base detect_lora detect_full


## 4. Segmentação

Os experimentos abaixo só rodam quando existir `data/segmentation/data.yaml`, que será produzido após fecharmos as máscaras/revisão. Se ainda não existir, o runner registra `missing_data` e pula sem quebrar a bateria.


In [ ]:
seg = Path('data/segmentation/data.yaml')
print('Dataset de segmentação pronto:', seg.exists(), '-', seg)
if seg.exists():
    subprocess.run([sys.executable, 'scripts/train_experiments.py', '--group', 'smoke_tests', '--only', 'segment_yolo26n_smoke'], check=True)


In [ ]:
if Path('data/segmentation/data.yaml').exists():
    subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'segment_base', 'segment_lora', 'segment_full'], check=True)
else:
    print('Segmentação ainda não executada: gere primeiro data/segmentation/data.yaml')


## 5. Tabela consolidada

O runner grava hiperparâmetros, parâmetros totais/treináveis, VRAM, tempo, checkpoints, métricas e velocidade em `workspace/training_runs/experiment_summary.csv/json`.


In [ ]:
import pandas as pd
summary = Path('workspace/training_runs/experiment_summary.csv')
if summary.exists():
    df = pd.read_csv(summary)
    display(df)
else:
    print('A tabela aparecerá após o primeiro experimento.')


## 6. Retomar após desconexão

Exemplo para continuar o full fine-tune de detecção usando `last.pt`, quando disponível.


In [ ]:
# Descomente quando necessário:
# !python scripts/train_experiments.py --only detect_full --resume
